# Notebook for selecting proteins for experimental validation

In [ ]:
import pandas as pd 
import os
import itertools
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
repo_path = os.path.dirname(os.getcwd())
repo_path = repo_path.split("WP2_Sweet_Protein_Discovery")[0] + "/WP2_Sweet_Protein_Discovery"

# Set rewrite to True if you want to overwrite the existing files, otherwise set it to False
rewrite = False

# Import sequence identity function from utils.py
sys.path.append(repo_path)
from utils import sequence_identity

peptide_name_abbreviations = {
    "MNEI_1IV7": "MN",
    "Honey_truffle_sweetener": "HTS",
    "Honey_truffle_sweetener_native": "HTS_native"
}
final_cols = ['original_id', 'protein_name', 'sequence', 'cysteine_count', 'disulfide_bridges']

# Load 0_de_novo_candidates.tsv, 0_foldseek_candidates.tsv and all_candidates.tsv
all_candidates = pd.read_csv(os.path.join(repo_path, "8_protein_selection", "1_merged_candidates.tsv"), sep = "\t")

# The following protein is has an identical sequence to another protein
all_candidates = all_candidates[all_candidates["target_id"] != "OneKP_scaffold-IXLH-2105913-Polypodium_hesperium-B_type"]

# Using results from 0.5 sequence identity clustering using MMseqs2 
seq_id = "0.5"

# MNEI
# Read clustered sequences and get the highest similarity hit for each cluster.
# Removing cluster containing the de novo sequences (fixed_ and free_) to avoid biasing the selection towards de novo candidates.
mnei_clusters_df = pd.read_csv(f"{repo_path}/8_protein_selection/2_mmseqs_clustered_hits/MNEI_1IV7/MNEI_1IV7_{seq_id}_cluster.tsv", sep="\t", header=None, names=["cluster_id", "target_id"])
mnei_clusters_df = mnei_clusters_df.drop_duplicates()
mnei_clusters_df = mnei_clusters_df.merge(all_candidates, on = "target_id", how = "inner")
mnei_clusters_df = mnei_clusters_df.sort_values(by=['cluster_id', 'avg_sweet_res_similarity'], ascending=[True, False])
mnei_clusters_df['representative_hit'] = mnei_clusters_df.groupby('cluster_id')['target_id'].transform('first')
mnei_clusters_df = mnei_clusters_df[~mnei_clusters_df["representative_hit"].str.startswith(("fixed", "free"))]
mnei_clusters_df = mnei_clusters_df.drop(columns=['cluster_id'])
print("MNEI has " + str(mnei_clusters_df["representative_hit"].nunique()) + " unique representative hits.")

# Selecting the top 10 clusters based on avg_sweet_res_similarity and keeping only the representative hit for each cluster
# based on the similarity of sweet residues
mnei_selected_df = mnei_clusters_df.sort_values(by='avg_sweet_res_similarity', ascending=False).groupby('representative_hit').head(1).head(10)
mnei_selected_df["rank"] = mnei_selected_df["avg_sweet_res_similarity"].rank(ascending=False)

# HTS
# Read clustered sequences and get the highest similarity hit for each cluster.
hts_clusters_df = pd.read_csv(f"{repo_path}/8_protein_selection/2_mmseqs_clustered_hits/Honey_truffle_sweetener/Honey_truffle_sweetener_{seq_id}_cluster.tsv", sep="\t", header=None, names=["cluster_id", "target_id"])
hts_clusters_df = hts_clusters_df.drop_duplicates()
hts_clusters_df = hts_clusters_df.merge(all_candidates, on = "target_id", how = "inner")
hts_clusters_df = hts_clusters_df.sort_values(by=['cluster_id', 'avg_sweet_res_similarity'], ascending=[True, False])
# Manually add Polypodium glycrrhiza hit
polypodium_glycyrrhiza_row = hts_clusters_df[hts_clusters_df["target_id"] == "OneKP_scaffold-CJNT-2007765-Polypodium_glycyrrhiza"].copy()
hts_clusters_df['representative_hit'] = hts_clusters_df.groupby('cluster_id')['target_id'].transform('first')
hts_clusters_df = hts_clusters_df.drop(columns=['cluster_id'])
print("Honey_truffle_sweetener has " + str(hts_clusters_df["representative_hit"].nunique()) + " unique representative hits.")
hts_selected_df = hts_clusters_df.sort_values(by='avg_sweet_res_similarity', ascending=False).groupby('representative_hit').head(2)
hts_selected_df = pd.concat([hts_selected_df, polypodium_glycyrrhiza_row]).drop_duplicates(subset=["target_id"])
hts_selected_df.loc[hts_selected_df["query"] == "Honey_truffle_sweetener_native", "avg_sweet_res_similarity"] = 1.0
hts_selected_df["rank"] = hts_selected_df["avg_sweet_res_similarity"].rank(ascending=False)

# Concatenate the selected candidates for MNEI and Thaumatin and save to a new dataframe
final_representatives = pd.concat([mnei_selected_df, hts_selected_df])
final_representatives["rank"] = final_representatives["rank"].astype(int)
final_representatives["protein_name"] = final_representatives.apply(lambda row: peptide_name_abbreviations[row["query"]] + "_" + str(int(row["rank"])), axis = 1)
final_representatives = final_representatives.rename(columns={"target_id": "original_id"})
final_representatives = final_representatives[final_cols].copy()
final_representatives

MNEI has 33 unique representative hits.
Honey_truffle_sweetener has 6 unique representative hits.


,original_id,protein_name,sequence,cysteine_count,disulfide_bridges
110,OneKP_scaffold-AALA-2005416-Meliosma_cuneifolia,MN_1,GGWTKIEDVKKNEEIQDLGRFSVEKFNKNKEQYQAKGRNKHDLTFS...,0.0,[]
0,A0A8B8NA16,MN_2,GEWNPIKDVRNPHVKEIGEFAVTEFNKDPQHGADLQFQSVIKGETQ...,0.0,[]
148,A0A7N1A7N7,MN_3,GGWQPIKNLDDKEIRELAKFALTEHRKLGGPRLEFERVLEGEQQVV...,0.0,[]
1,OneKP_scaffold-KWGC-2062037-Crassula_perforata,MN_4,GGLMEIEDVEHNKEVQDLGKFSVAQYNNKLAEKNKLLEFVQVVSAE...,0.0,[]
2,A0A7N0UIJ2,MN_5,GGWKKISNLTDPSIIAIAEFAVEEAAKVNTHTSSLTFKRIRSGKYQ...,0.0,[]
3,A0A9Q0G024,MN_6,GGWEPVSNHDPHVREVGEFAVRAYNEQGKIKGAGLKFESVLKAARQ...,0.0,[]
159,A0A0D3FFA6,MN_7,GSWEPVDINDPHVQELGRWAVAEEDRGVAAGGLTFERVTDGEKQVV...,0.0,[]
4,A0ABP0U637,MN_8,GAPKNIPNSNSVEIRELANFAVSQYNSGKNVVHKLSFSKVVSAREQ...,1.0,[]
169,A0AB40C3D9,MN_9,GGRTEIKNVKSNKMVQSLGKFAVEEYDKGVKGGSNAAALVSFSSVI...,0.0,[]
70,OneKP_scaffold-YKFU-2009113-Peltoboykinia_wata...,MN_10,GGRTKIRDVKSNQEVQNLGKFSVEEYNKQSRQNGGLLMFKEVVEAE...,0.0,[]


# De novo designed sequences for MNEI

In [2]:
# Sort based on avg_sweet_res_similarity and remove sequences with high sequence identity
de_novo_candidates = pd.read_csv(os.path.join(repo_path, "8_protein_selection", "0_de_novo_candidates.tsv"), sep = "\t")
de_novo_candidates = de_novo_candidates.sort_values(by="avg_sweet_res_similarity", ascending=False)
de_novo_candidates["disulfide_bridges"] = None

seq_ids = []
for (index1, row1), (index2, row2) in itertools.combinations(de_novo_candidates.iterrows(), 2):
    candidate_seq = row1["sequence"]
    representative_seq = row2["sequence"]
    identity = sequence_identity(candidate_seq, representative_seq)
    seq_ids.append([row1["id"], row2["id"], identity])

# Removing sequences with high sequence identity (>= 95%) to other candidates to ensure we have a diverse set of candidates.
seq_ids_df = pd.DataFrame(seq_ids, columns=["id", "id_2", "identity"])
seq_ids_df = seq_ids_df[seq_ids_df["identity"] >= 0.95].copy()
ids_to_drop = seq_ids_df["id_2"].unique()
de_novo_candidates = de_novo_candidates[~de_novo_candidates["id"].isin(ids_to_drop)] 

# Choose top 5 candidates from fixed and free each based on avg_sweet_res_similarity, ensuring that there are no duplicate sequences in the final selection.
fixed_candidates = de_novo_candidates[de_novo_candidates["id"].str.startswith("fixed_")].copy()
fixed_candidates = fixed_candidates.sort_values(by="avg_sweet_res_similarity", ascending=False).head(5)["id"].to_list()
free_candidates = de_novo_candidates[de_novo_candidates["id"].str.startswith("free_")].copy()
free_candidates = free_candidates.sort_values(by="avg_sweet_res_similarity", ascending=False).head(5)["id"].to_list()

# Merge the two de novo lists and show the selected de novo candidates.
free_candidates_df = de_novo_candidates[de_novo_candidates["id"].isin(free_candidates)].copy()
fixed_candidates_df = de_novo_candidates[de_novo_candidates["id"].isin(fixed_candidates)].copy()
final_de_novo_candidates_df = pd.concat([free_candidates_df, fixed_candidates_df])
final_de_novo_candidates_df = final_de_novo_candidates_df.sort_values(by="avg_sweet_res_similarity", ascending=False)
final_de_novo_candidates_df["rank"] = final_de_novo_candidates_df["avg_sweet_res_similarity"].rank(ascending=False)
final_de_novo_candidates_df["rank"] = final_de_novo_candidates_df["rank"].astype(int)
final_de_novo_candidates_df["original_id"] = final_de_novo_candidates_df["id"]

final_de_novo_candidates_df["protein_name"] = None
for index, row in final_de_novo_candidates_df.iterrows():
    protein_name = f"MN_de_novo_{str(row['rank'])}"
    final_de_novo_candidates_df.at[index, "disulfide_bridges"] = []
    final_de_novo_candidates_df.at[index, "protein_name"] = protein_name

final_de_novo_candidates_df = final_de_novo_candidates_df[final_cols].copy()
final_de_novo_candidates_df

,original_id,protein_name,sequence,cysteine_count,disulfide_bridges
0,fixed_02_MNEI_1IV7_8,MN_de_novo_1,MEEEIIPVDEFTQELGKFAVEEENKRGQFGKLTFDKVLEPCIKKTI...,1,[]
3,fixed_015_MNEI_1IV7_3,MN_de_novo_2,SEERVVPVDDFTQELGKFAVEEENKRGQFGKLTFDKVLAPCIKHTI...,1,[]
5,fixed_015_MNEI_1IV7_15,MN_de_novo_3,MEKEVIPIDEFTQELGKFAVEEENKKGQFGKLTFDKVLEPCIKYTI...,1,[]
6,fixed_02_MNEI_1IV7_18,MN_de_novo_4,METEIIPIGPFTEELGRFAVEEENKRGQFGKLTFDKVLEPCIKHTI...,1,[]
7,fixed_025_MNEI_1IV7_15,MN_de_novo_5,MEKEVLPIDEFTQELGKFAVEEENKKGQFGKLTFDKVLEPCIKETI...,1,[]
33,free_03_MNEI_1IV7_18,MN_de_novo_6,MEEEIIPVDEFTQELGKFAVEEQNKEGKFGELTFDKVVEPVIKKTI...,0,[]
34,free_025_MNEI_1IV7_19,MN_de_novo_7,MKREVVPVGPETQELGKFAVEEVNKDGKFGTLTFDEVLPPVIKETI...,0,[]
36,free_01_MNEI_1IV7_15,MN_de_novo_8,MEEEIVPVDEFTQELGKFAVEEQNKKGEFGKLTFDKVVEPVIKHTI...,0,[]
37,free_025_MNEI_1IV7_7,MN_de_novo_9,MEKEIVPVNEETQELGKFAVEQENKNGKFGKLTFDKVIEPVTKTTL...,0,[]
38,free_025_MNEI_1IV7_14,MN_de_novo_10,AEKEIVPVGPFTQELGKFAVEEENKRGKFGKLEFDKVVEPVVKETL...,0,[]


In [ ]:
# Concatenate the selected representatives from both approaches
quote_proteins = pd.concat([final_representatives, final_de_novo_candidates_df]).reset_index(drop=True)

# Read original sequences
foldseek_uniprot_df = pd.read_csv(f"{repo_path}/3_foldseek_search/foldseek_UniProt_search.tsv", sep = "\t")
foldseek_uniprot_df["target"] = foldseek_uniprot_df["target"].apply(lambda x: x.split("|")[1] if "|" in x else x)
foldseek_onekp_df = pd.read_csv(f"{repo_path}/3_foldseek_search/foldseek_OneKP_search.tsv", sep = "\t")

# Keep only target and tseq
foldseek_uniprot_df = foldseek_uniprot_df[["target", "tseq"]].copy()
foldseek_onekp_df = foldseek_onekp_df[["target", "tseq"]].copy()

# Merge the two foldseek results to get the original sequences for all candidates
foldseek_df = pd.concat([foldseek_uniprot_df, foldseek_onekp_df]).drop_duplicates(subset=["target"]).copy()
quote_proteins = quote_proteins.merge(foldseek_df, left_on="original_id", right_on="target", how="left").drop(columns=["target"]).rename(columns={"tseq": "full_sequence"})

# Peptide name = peptide_name_abbreviation + rank 
rename_cols = {
    "protein_name": "Protein_ID",
    "cysteine_count": "Cysteine_Count",
    "sequence": "Sequence",
    "disulfide_bridges": "Disulfide_Bridges",
    "original_id": "Original_ID",
    "full_sequence": "Full Sequence"
}

quote_proteins = quote_proteins.rename(columns = rename_cols)
quote_proteins = quote_proteins[rename_cols.values()]

# Save fasta file of the selected candidates for synthesis
if rewrite:
    fasta_path = os.path.join(repo_path, "8_protein_selection", "3_quote_full_sequences.fasta")
    if os.path.exists(fasta_path):
        os.remove(fasta_path)
    for index, row in quote_proteins.iterrows():
        fasta_header = f">{row['Protein_ID']}"
        if pd.isna(row["Full Sequence"]):
            fasta_sequence = row["Sequence"]
        else:
            fasta_sequence = row["Full Sequence"]
        fasta_entry = f"{fasta_header}\n{fasta_sequence}\n"
        with open(fasta_path, "a") as fasta_file:
            fasta_file.write(fasta_entry)

# Add DeepLoc 2 predictions to the resulting dataframe
DeepLoc 2 was run using the following command:
```
deeploc2 -f 3_quote_full_sequences.fasta -m Fast -o ./3_deeploc2_results
```

In [ ]:
deeploc_res = f"{repo_path}/8_protein_selection/3_deeploc2_results/results_20260617-111028.csv"
deeploc_df = pd.read_csv(deeploc_res)
deeploc_df = deeploc_df[deeploc_df["Protein_ID"].isin(quote_proteins["Protein_ID"])]
deeploc_df = deeploc_df[["Protein_ID", "Localizations", "Membrane types", "Signals"]]
deeploc_df = deeploc_df.rename(columns={
    "Membrane types": "Membrane_Types"
})

# Merge to quote_proteins to get the localization information for each candidate
quote_proteins_final = quote_proteins.merge(deeploc_df, on="Protein_ID", how="inner")
quote_proteins_final = quote_proteins_final[~((quote_proteins_final["Protein_ID"].str.startswith("TH_")) & (quote_proteins_final["Protein_ID"] != "TH_1"))].copy()
if rewrite:
    quote_proteins_final[["Protein_ID", "Original_ID", "Sequence"]].to_csv(os.path.join(repo_path, "8_protein_selection", "3_genscript_quote_id_mapping.csv"), index = False)
quote_proteins_final_df = quote_proteins_final.drop(columns=["Original_ID"])

quote_proteins_final_df["Expressed_Sequence"] = ""
for index, row in quote_proteins_final_df.iterrows():
    # If the length of Original Sequence differs from full length sequence by less then 5 amino acids,
    # use the full length sequence for synthesis, otherwise use the original sequence (which is the sequence of the aligned region to the query).
    if pd.isna(row["Full Sequence"]):
        quote_proteins_final_df.at[index, "Expressed_Sequence"] = row["Sequence"]
    else:
        if abs(len(row["Full Sequence"]) - len(row["Sequence"])) <= 5:
            quote_proteins_final_df.at[index, "Expressed_Sequence"] = row["Full Sequence"]
        else:
            quote_proteins_final_df.at[index, "Expressed_Sequence"] = row["Sequence"]

# Drop all TH_ hits except TH_1
if rewrite:
    quote_proteins_final_df[["Protein_ID", "Expressed_Sequence"]].to_csv(os.path.join(repo_path, "8_protein_selection", "3_genscript_quote.csv"), index = False)

quote_proteins_final_df

,Protein_ID,Cysteine_Count,Sequence,Disulfide_Bridges,Full Sequence,Localizations,Membrane_Types,Signals,Expressed_Sequence
0,MN_1,0.0,GGWTKIEDVKKNEEIQDLGRFSVEKFNKNKEQYQAKGRNKHDLTFS...,[],FRYSALIFFLLVCISSISVDGFGRKVGGWTKIEDVKKNEEIQDLGR...,Extracellular,Soluble,Signal peptide,GGWTKIEDVKKNEEIQDLGRFSVEKFNKNKEQYQAKGRNKHDLTFS...
1,MN_2,0.0,GEWNPIKDVRNPHVKEIGEFAVTEFNKDPQHGADLQFQSVIKGETQ...,[],MRRQLHLLLLLLLLLAALLLSALAIGEWNPIKDVRNPHVKEIGEFA...,Extracellular,Soluble,Signal peptide,GEWNPIKDVRNPHVKEIGEFAVTEFNKDPQHGADLQFQSVIKGETQ...
2,MN_3,0.0,GGWQPIKNLDDKEIRELAKFALTEHRKLGGPRLEFERVLEGEQQVV...,[],MAYSKTSILLLMAVAFTLASSAAPAEVERGPRLGGWQPIKNLDDKE...,Extracellular,Soluble,Signal peptide,GGWQPIKNLDDKEIRELAKFALTEHRKLGGPRLEFERVLEGEQQVV...
3,MN_4,0.0,GGLMEIEDVEHNKEVQDLGKFSVAQYNNKLAEKNKLLEFVQVVSAE...,[],GGKLGGLMEIEDVEHNKEVQDLGKFSVAQYNNKLAEKNKLLEFVQV...,Nucleus,Soluble,Nuclear localization signal,GGKLGGLMEIEDVEHNKEVQDLGKFSVAQYNNKLAEKNKLLEFVQV...
4,MN_5,0.0,GGWKKISNLTDPSIIAIAEFAVEEAAKVNTHTSSLTFKRIRSGKYQ...,[],MASFKFGTIFIATLAWTLLLVSNGQIHVDEGKHPPVGGWKKISNLT...,Extracellular,Soluble,Signal peptide,GGWKKISNLTDPSIIAIAEFAVEEAAKVNTHTSSLTFKRIRSGKYQ...
5,MN_6,0.0,GGWEPVSNHDPHVREVGEFAVRAYNEQGKIKGAGLKFESVLKAARQ...,[],MANHCLFLCLLLLAAATVAYGGSLGGWEPVSNHDPHVREVGEFAVR...,Extracellular,Soluble,Signal peptide,GGWEPVSNHDPHVREVGEFAVRAYNEQGKIKGAGLKFESVLKAARQ...
6,MN_7,0.0,GSWEPVDINDPHVQELGRWAVAEEDRGVAAGGLTFERVTDGEKQVV...,[],MTMRTSSLLLAAVAVVAIVAGATAATVGSWEPVDINDPHVQELGRW...,Extracellular,Soluble,Signal peptide,GSWEPVDINDPHVQELGRWAVAEEDRGVAAGGLTFERVTDGEKQVV...
7,MN_8,1.0,GAPKNIPNSNSVEIRELANFAVSQYNSGKNVVHKLSFSKVVSAREQ...,[],MMKKAVLMLSIIMSLVLLLLLLQSSSSSSLAAERTMVGAPKNIPNS...,Lysosome/Vacuole,Soluble,Signal peptide,GAPKNIPNSNSVEIRELANFAVSQYNSGKNVVHKLSFSKVVSAREQ...
8,MN_9,0.0,GGRTEIKNVKSNKMVQSLGKFAVEEYDKGVKGGSNAAALVSFSSVI...,[],MASRLAPVLLLVLCAIASSATASIVGGRTEIKNVKSNKMVQSLGKF...,Extracellular,Soluble,Signal peptide,GGRTEIKNVKSNKMVQSLGKFAVEEYDKGVKGGSNAAALVSFSSVI...
9,MN_10,0.0,GGRTKIRDVKSNQEVQNLGKFSVEEYNKQSRQNGGLLMFKEVVEAE...,[],GRFVGGRTKIRDVKSNQEVQNLGKFSVEEYNKQSRQNGGLLMFKEV...,Nucleus,Soluble,NaN,GRFVGGRTKIRDVKSNQEVQNLGKFSVEEYNKQSRQNGGLLMFKEV...
